# CA2 – Online Retail: Sales Analysis and Interactive Dashboard

**Module:** Data Visualisation and Communication &nbsp;|&nbsp; **Lecturer:** Taufique Ahmed  
**Student:** Matheus Vinicios de Castilho &nbsp;|&nbsp; **Student number:** 2023336

**GitHub repository:** https://github.com/Matheus-Castilho/CA2-DV 

> **How to run this notebook:** place `Online Retail.xlsx` in the same folder as this notebook (the data is not included in the repository), then select *Kernel → Restart Kernel and Run All Cells*.  
> Libraries used: `pandas`, `matplotlib`, `openpyxl` (to read Excel) and `ipywidgets` (for the dashboard).

<a id="intro"></a>
## 1. Introduction

 #TODO

<a id="setup"></a>
## 2. Setup
mporting the libraries used (pandas for data handling, matplotlib for charts) and setting a consistent chart style

In [17]:
import pandas as pd
import matplotlib.pyplot as plt

# Show all columns and format decimals neatly
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Same chart style across the whole notebook
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

<a id="load"></a>
## 3. Loading the data

`InvoiceNo` and `StockCode` are read as **text** because some of their values contain letters (for example `C536379` or `85123A`). Without this, pandas would mix numbers and text in the same column.

The original data is kept unchanged in `df_raw`, and all cleaning is done on a copy called `df`. This makes it possible to compare before and after cleaning. Loading the Excel file can take around 30 seconds.

In [18]:
# Read InvoiceNo and StockCode as text because some values contain letters
df_raw = pd.read_excel('Online Retail.xlsx', dtype={'InvoiceNo': str, 'StockCode': str})

# Work on a copy so the original data stays unchanged
df = df_raw.copy()

print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

Rows: 541,909  |  Columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


<a id="quality"></a>
## 4. Data quality assessment

Before cleaning, the data is checked for problems: wrong data types, missing values, duplicates, invalid records (such as cancellations and negative quantities), items that are not real products, and outliers. **Nothing is removed in this section.** Each issue is only measured here, and the decisions about how to handle it are taken in the cleaning section.

### 4.1 Structure and data types

The table below describes each column and the data type it should have. `df.info()` is then used to compare this with how pandas actually read the data.

| Column | Meaning | Expected type |
|---|---|---|
| `InvoiceNo` | Invoice number. Invoices starting with **C** are cancellations | Text |
| `StockCode` | Product code | Text |
| `Description` | Product name | Text |
| `Quantity` | Units of the product on this invoice line | Whole number |
| `InvoiceDate` | Date and time of the transaction | Date/time |
| `UnitPrice` | Price per unit in pounds sterling (£) | Decimal |
| `CustomerID` | Customer number | ID (text) |
| `Country` | Country where the customer is based | Text |

In [19]:
# Column names, data types and number of non-missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(3)
memory usage: 45.7+ MB


**Observations:**
- The dataset has **541,909 rows and 8 columns**.
- `InvoiceDate` is already read as a date/time, so sales can be grouped by month, weekday and hour without any conversion.
- `CustomerID` is stored as a decimal number (`float64`). This happens because the column has missing values. It is an identifier, not something to calculate with, so it will be converted to text during cleaning.
- `Description` and `CustomerID` have fewer non-null values than the other columns, so they contain missing data. This is checked in the next section.

### 4.2 Descriptive statistics

Summary statistics (count, mean, median, minimum and maximum) give a quick overview of the numeric columns and highlight values that cannot be correct. Counting the unique values in each column shows how many invoices, products, customers and countries the data covers.

In [20]:
# Summary statistics for the numeric and date columns
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,"541,909.00",541909,"541,909.00","406,829.00"
mean,9.55,2011-07-04 13:34:57.156386,4.61,"15,287.69"
min,"-80,995.00",2010-12-01 08:26:00,"-11,062.06","12,346.00"
25%,1.00,2011-03-28 11:34:00,1.25,"13,953.00"
50%,3.00,2011-07-19 17:17:00,2.08,"15,152.00"
75%,10.00,2011-10-19 11:27:00,4.13,"16,791.00"
max,"80,995.00",2011-12-09 12:50:00,"38,970.00","18,287.00"
std,218.08,NaN,96.76,"1,713.60"


In [21]:
# Number of different values in each column
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

**Observations:**
- `Quantity` ranges from **−80,995 to +80,995**. Both extremes are the same size, which suggests one very large order that was entered and then cancelled. This is checked in the outliers section.
- `UnitPrice` has a **negative minimum (−£11,062.06)**, which cannot be a real price. Rows like this are accounting adjustments, not sales.
- The **median** quantity is 3 units and the median price is about £2, but both **means** are higher. This shows the data is **right-skewed**: most invoice lines are small, with a few very large ones pulling the average up. For this reason the median is a better measure of a "typical" order line.
- The data covers about **25,900 invoices, 4,070 products, 4,372 customers and 38 countries**, from December 2010 to December 2011.
- `Description` has more unique values than `StockCode` (4,223 vs 4,070), so some products appear under slightly different names. To avoid splitting one product into several rows, product charts will group by `StockCode` and show one name per code.

### 4.3 Missing values

Missing values are counted for every column, and only the columns that have gaps are shown. For each one, the next step is to check whether the affected rows are real sales, because that decides whether they should be kept or removed.

In [22]:
# Count missing values per column and show them as a percentage of all rows
missing = pd.DataFrame({
    'Missing values': df.isna().sum(),
    '% of rows': (df.isna().mean() * 100).round(2)
})
missing[missing['Missing values'] > 0]

,Missing values,% of rows
Description,1454,0.27
CustomerID,135080,24.93


In [23]:
# Are the rows without a description real sales?
no_desc = df[df['Description'].isna()]
print('Unit prices of rows with no description:', no_desc['UnitPrice'].unique())
print('Of these, rows also missing CustomerID:', no_desc['CustomerID'].isna().sum(), 'of', len(no_desc))

Unit prices of rows with no description: [0.]
Of these, rows also missing CustomerID: 1454 of 1454


**Observations:**
- **CustomerID is missing in about 25% of rows** (135,080). These are most likely guest purchases or orders not linked to a customer account. They are still real sales, so removing them would understate revenue by roughly a quarter. They will be **kept and labelled as `Guest`**, and left out only when analysing individual customers.
- **All 1,454 rows without a description have a unit price of £0** and no customer. They cannot be real sales (most likely internal stock adjustments), so they will be removed by the zero-price rule during cleaning.

### 4.4 Duplicates

A row is treated as a duplicate when it is identical to another row in **every** column, including the invoice, product, quantity, price and time.

In [24]:
dup_count = df.duplicated().sum()
print(f'Exact duplicate rows: {dup_count:,} ({dup_count / len(df) * 100:.2f}% of rows)')

# Show a few examples: keep=False marks every copy, not just the second one
df[df.duplicated(keep=False)].sort_values(['InvoiceNo', 'StockCode']).head(6)

Exact duplicate rows: 5,268 (0.97% of rows)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom


**Observations:**
- There are **5,268 exact duplicate rows** (about 1% of the data). The examples show the same product, quantity and price on the same invoice at the same minute, so they are most likely double entries. They will be **removed** so the same sale is not counted twice.
- *Limitation:* a customer could legitimately add the same item twice to one invoice, but the data has no line number to tell these cases apart. Removing them is the safer choice, because it avoids overstating revenue.

### 4.5 Cancellations and invalid records

Invoices that start with the letter **C** are cancellations. Rows with a quantity or unit price of zero or less are also checked, because a real sale must have at least one unit and a positive price.

In [25]:
# Invoices starting with 'C' are cancellations
is_cancelled = df['InvoiceNo'].str.startswith('C')
cancelled_invoices = df.loc[is_cancelled, 'InvoiceNo'].nunique()
all_invoices = df['InvoiceNo'].nunique()

print(f'Cancelled rows: {is_cancelled.sum():,}')
print(f'Cancelled invoices: {cancelled_invoices:,} of {all_invoices:,} '
      f'({cancelled_invoices / all_invoices * 100:.1f}% of all invoices)')
print(f'Rows with Quantity <= 0: {(df["Quantity"] <= 0).sum():,}')
print(f'Rows with UnitPrice <= 0: {(df["UnitPrice"] <= 0).sum():,}')

Cancelled rows: 9,288
Cancelled invoices: 3,836 of 25,900 (14.8% of all invoices)
Rows with Quantity <= 0: 10,624
Rows with UnitPrice <= 0: 2,517


In [26]:
# A few examples of cancelled rows
df[is_cancelled].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,"14,527.00",United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,"15,311.00",United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,"17,548.00",United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,"17,548.00",United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,"17,548.00",United Kingdom


**Observations:**
- **About 15% of invoices are cancellations** (3,836 of 25,900). This is important for the business on its own, because cancellations and returns cost money in handling and postage. Cancelled invoices are not completed sales, so they will be **removed** from the revenue analysis.
- Every cancelled row has a negative quantity. The rows with a quantity of zero or less include these cancellations plus other stock corrections.
- Rows with a price of zero or less are free items, damaged stock or accounting adjustments (for example the *"Adjust bad debt"* entries). They cannot represent a real sale, so they will also be **removed**.

### 4.6 Codes that are not products

Product codes normally start with five digits (for example `85123A`). Listing the codes that do not follow this pattern and reading their descriptions shows that several of them are **not merchandise**: postage, fees, discounts and manual adjustments.

In [27]:
# Codes for postage, fees, discounts and manual adjustments (found by checking codes that are not 5 digits)
non_product_codes = ['POST', 'DOT', 'M', 'm', 'C2', 'D', 'S', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'B']

non_product = df[df['StockCode'].isin(non_product_codes)]
(non_product
 .groupby('StockCode')
 .agg(Description=('Description', 'first'), Rows=('Description', 'size'))
 .sort_values('Rows', ascending=False))

,Description,Rows
StockCode,,
POST,POSTAGE,1256
DOT,DOTCOM POSTAGE,710
M,Manual,571
C2,CARRIAGE,144
D,Discount,77
S,SAMPLES,63
BANK CHARGES,Bank Charges,37
AMAZONFEE,AMAZON FEE,34
CRUK,CRUK Commission,16


**Observations:**
- There are **2,912 rows** with these codes, covering postage (`POST`, `DOT`, `C2`), fees (`AMAZONFEE`, `BANK CHARGES`, `CRUK`), discounts (`D`), samples (`S`) and manual or bad-debt adjustments (`M`, `m`, `B`).
- If these were kept, *POSTAGE* would appear among the "top products", which would mislead managers. They will be **removed** before the product analysis.
- Other unusual codes, such as gift vouchers (`gift_0001_...`) and a few `DCGS...` items, are **kept** because they are real items sold to customers.